# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My lane: Refresh / Content Opportunity Scoring.** I'm picking this lane because I want real model-building practice, not just exploratory analysis — this lane has me build a transparent baseline, then train and compare actual models (logistic regression, decision tree, random forest) against it, which is the full supervised-ML loop I want to learn this internship. It's also the lane with the most to learn from: the starter pipeline (`scripts/01`–`05`) already builds this exact lane end-to-end, with real results already sitting in `outputs/model_report.md`, so I can read working code before extending it myself.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

LANE_NAME = "Refresh / Content Opportunity Scoring"
LANE_TYPE = "Core lane (predefined)"
print(f"Lane: {LANE_NAME} ({LANE_TYPE})")

Lane: Refresh / Content Opportunity Scoring (Core lane (predefined))


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** which pages a reviewer should look at first for refresh, given limited reviewer hours each week.

**Who acts, and how:** a content strategist works down a ranked review queue and decides, per page, whether to refresh, expand, protect, prune, or monitor it — the model doesn't take the action, the person does.

**Cost of a wrong call:** a false positive burns scarce reviewer hours on a page that didn't need attention; a false negative lets a genuinely declining, high-demand page keep losing visibility unreviewed. Both are costly because review capacity is small relative to the inventory — today, 58.1% of pages trip at least one "worth reviewing" flag (see code below), far more than any team can work through by hand.

**Why ML, not just a rule:** the starter pipeline already shows the pattern is real but too tangled for a fixed rule alone — a learned model finds roughly 37 of the top 50 truly-worth-reviewing pages (precision@50 = 0.740), against about 12 for the fixed rule (0.240). Freshness, position, trend, and volume interact in ways a simple if/then doesn't capture well.

**One-paragraph frame:** For a content strategist deciding which pages to review first for refresh, we will build a ranked review queue from the starter content-performance data, scoring refresh opportunity, measured by precision@K on that ranked list. A wrong call costs either wasted reviewer hours or a missed declining page. A plain rule already over-flags 58% of the inventory, which isn't enough — the signals are real but too tangled to rank by hand. We will claim only observed and decision-support results.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

stale_visible = (df.days_since_last_update >= 180) & (df.impressions_90d >= 500)
declining_demand = (df.trend_direction == "down") & (df.impressions_90d >= 100)
thin_visible = (df.word_count > 0) & (df.word_count < 1200) & (df.impressions_90d >= 250)
page_one_decay = (df.avg_position > 0) & (df.avg_position <= 10) & (df.content_age_days >= 180)

any_flag = stale_visible | declining_demand | thin_visible | page_one_decay
print(f"Pages tripping at least one review-worthy flag today: {any_flag.sum():,} of {len(df):,} ({100*any_flag.mean():.1f}%)")
print("-> far more than any reviewer can look at by hand each week; a ranked queue is what makes the review capacity usable.")

Pages tripping at least one review-worthy flag today: 17,430 of 30,000 (58.1%)
-> far more than any reviewer can look at by hand each week; a ranked queue is what makes the review capacity usable.


## 3. Quick look at the data (2-3 real numbers)

Loaded the starter CSV (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32 pseudonymized clients) and pulled three numbers that make this lane worth the next 7 weeks — see the code cell below for the exact filters and gotcha-safe handling (`avg_position > 0` excludes the 1,205 "no data" rows; `trend_direction`/`trend_pct` are read here only as a filter, never as a model feature).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,} across {df.client_id.nunique()} pseudonymized clients\n")

declining_demand = (df.trend_direction == "down") & (df.impressions_90d >= 100)
print(f"1) 'Declining with demand' (trend down AND >=100 impressions/90d): "
      f"{declining_demand.sum():,} pages ({100*declining_demand.mean():.1f}%) "
      f"-> real visible traffic being lost, not noise.")

page_one_decay = (df.avg_position > 0) & (df.avg_position <= 10) & (df.content_age_days >= 180)
print(f"2) 'Page-one decay risk' (avg_position 1-10, content_age_days>=180; avg_position>0 excludes the 1,205 no-data rows): "
      f"{page_one_decay.sum():,} pages ({100*page_one_decay.mean():.1f}%) "
      f"-> valuable ranking real estate aging without a refresh.")

print("3) The starter pipeline (scripts/01-05, this same 30k-row slice) already shows a learned model "
      "beating a fixed rule at review time: precision@50 of 0.740 (random forest) vs 0.240 (baseline rules) "
      "-- source: outputs/model_report.md. On a 50-page weekly review batch, that's ~37 correct flags vs ~12.")

Rows: 30,000 across 32 pseudonymized clients

1) 'Declining with demand' (trend down AND >=100 impressions/90d): 13,152 pages (43.8%) -> real visible traffic being lost, not noise.
2) 'Page-one decay risk' (avg_position 1-10, content_age_days>=180; avg_position>0 excludes the 1,205 no-data rows): 7,076 pages (23.6%) -> valuable ranking real estate aging without a refresh.
3) The starter pipeline (scripts/01-05, this same 30k-row slice) already shows a learned model beating a fixed rule at review time: precision@50 of 0.740 (random forest) vs 0.240 (baseline rules) -- source: outputs/model_report.md. On a 50-page weekly review batch, that's ~37 correct flags vs ~12.


## 4. Careful words: what I can and can't claim

**What this work can say:** observed patterns in this 90-day snapshot (e.g., which pages carry a "declining + demand" or "page-one decay" flag today); directional signal from comparing a learned ranking against the fixed rule on the same historical slice; decision-support value — a ranked queue, with inspectable reason codes, that helps a reviewer spend limited hours on the more promising pages first.

**What this work will never say:** that refreshing a page *causes* it to recover (that needs an experiment, not this data); that the score reflects Google's actual ranking algorithm; that a flagged page is a genuine decline rather than seasonality, consolidation, or noise, without checking; or anything derived from `trend_direction`/`trend_pct` used as a model feature rather than the label source it is. Pseudonymous IDs (`content_id`, `client_id`) are used for grouping only, never as features, and nothing raw (names, URLs, queries) leaves this notebook — confirmed in the code cell below.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
unsafe_markers = [c for c in df.columns if any(k in c.lower() for k in ["url", "domain", "title", "client_name", "query_text"])]
print("Columns that could carry raw private text:", unsafe_markers if unsafe_markers else "none found")
print("content_id example:", df.content_id.iloc[0], "| client_id example:", df.client_id.iloc[0])

Columns that could carry raw private text: none found
content_id example: content_304f48230142 | client_id example: client_f369cb89fc


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.